# Musical Instrument Classification with CNNs on Log-Mel Spectrograms

**TECHIN 513 — Digital Signal Processing**  
University of Washington, Master of Science in Technology Innovation

---

This notebook classifies dominant musical instruments (drum, guitar, piano, violin) from short audio clips using convolutional neural networks trained on log-mel spectrogram representations.

**Key result:** Four iterative CNN versions improved from a 0.4848 logistic-regression baseline to **0.7148 validation accuracy** (v4), with each revision introducing a specific architectural or training change.

| Version | Change Introduced | Val Accuracy |
|---------|-------------------|-------------|
| Baseline | LogReg on flattened spectrograms | 0.4848 |
| V1 | 3-layer CNN, Flatten + Dense | 0.6806 |
| V2 | + BatchNorm, + GlobalAveragePooling, 4 conv blocks | 0.6825 |
| V3 | Back to 3 conv + Flatten, + class weights | 0.7110 |
| **V4** | **+ Dropout 0.5, + ReduceLROnPlateau** | **0.7148** |

> **Note on v3 vs v4:** V3 produced more balanced per-class recall (especially violin), while v4 maximized headline accuracy at the cost of violin recall. See confusion matrices below.

**Dataset:** [Kaggle Musical Instrument Sound Dataset](https://www.kaggle.com/datasets/soumendraprasad/musical-instruments-sound-dataset) (4 classes, ~2600 clips)  
**Report:** See the accompanying ACL-format paper for full analysis.

## 1. Setup & Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 513
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

## 2. Dataset Preparation

The [Kaggle Musical Instrument Sound Dataset](https://www.kaggle.com/datasets/soumendraprasad/musical-instruments-sound-dataset) contains labeled `.wav` files across instrument classes. We use four classes that were practical for training and demo: **drum, guitar, piano, violin**.

> Download the dataset and update `DATASET_DIR` / `TRAIN_AUDIO_DIR` to match your local paths.

In [ ]:
# ── Update these paths to match your environment ──
DATASET_DIR = "./instrument_data"
TRAIN_AUDIO_DIR = os.path.join(DATASET_DIR, "Train_submission", "Train_submission")

train_meta = pd.read_csv(os.path.join(DATASET_DIR, "Metadata_Train.csv"))
train_meta["path"] = train_meta["FileName"].apply(lambda x: os.path.join(TRAIN_AUDIO_DIR, x))

print("Train shape:", train_meta.shape)
print("\nClass counts:")
print(train_meta["Class"].value_counts())

In [ ]:
# Map labels and split
label_map = {
    "Sound_Guitar": "guitar",
    "Sound_Drum": "drum",
    "Sound_Violin": "violin",
    "Sound_Piano": "piano",
}

df = train_meta.copy()
df["label"] = df["Class"].map(label_map)

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)

le = LabelEncoder()
train_df["y"] = le.fit_transform(train_df["label"])
val_df["y"] = le.transform(val_df["label"])

num_classes = len(le.classes_)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Classes: {list(le.classes_)}")
print("\nTrain distribution:")
print(train_df["label"].value_counts())

## 3. Feature Extraction — Log-Mel Spectrograms

Each audio clip is converted to a **log-mel spectrogram**: a 2D time-frequency representation that CNNs can process like an image. Settings were chosen to balance frequency resolution and computational cost.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Sample rate | 22,050 Hz | Standard for music analysis (Nyquist at ~11 kHz) |
| Duration | 3.0 s | Fixed-length crops for uniform input |
| Mel bands | 64 | Sufficient for instrument timbre without overfitting |
| FFT size | 2048 | ~93 ms windows for good frequency resolution |
| Hop length | 512 | ~23 ms hops → 130 time frames per 3 s clip |

In [ ]:
# Audio + feature settings
SR = 22050
DURATION = 3.0
SAMPLES = int(SR * DURATION)

N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 512


def load_audio_fixed(path, sr=SR, samples=SAMPLES):
    """Load audio, pad or trim to fixed length."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    if len(y) < samples:
        y = np.pad(y, (0, samples - len(y)))
    else:
        y = y[:samples]
    return y


def wav_to_logmel(path, sr=SR, samples=SAMPLES,
                  n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Convert audio file to log-mel spectrogram."""
    y = load_audio_fixed(path, sr=sr, samples=samples)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    logmel = librosa.power_to_db(mel, ref=np.max)
    return logmel.astype(np.float32)

In [ ]:
# Visualize a sample spectrogram
sample_path = train_df.iloc[0]["path"]
sample_label = train_df.iloc[0]["label"]
spec = wav_to_logmel(sample_path)

print(f"Label: {sample_label}  |  Spectrogram shape: {spec.shape}")

plt.figure(figsize=(8, 4))
librosa.display.specshow(spec, sr=SR, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")
plt.title(f"Log-Mel Spectrogram: {sample_label}")
plt.tight_layout()
plt.show()

## 4. Baseline — Logistic Regression on Flattened Spectrograms

Before building CNNs, we establish a non-neural baseline: flatten each spectrogram into a 1D vector, standardize, and classify with logistic regression. This sets the floor at **0.4848** and justifies the move to convolutional models.

In [ ]:
# Build flattened feature matrices
def build_xy_flat(dataframe):
    X_list, y_list = [], []
    for _, row in dataframe.iterrows():
        spec = wav_to_logmel(row["path"])
        X_list.append(spec.flatten())
        y_list.append(row["y"])
    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.int64)

X_train_base, y_train_base = build_xy_flat(train_df)
X_val_base, y_val_base = build_xy_flat(val_df)

print("X_train:", X_train_base.shape, " X_val:", X_val_base.shape)

In [ ]:
# Baseline classifier
baseline_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED)
)
baseline_clf.fit(X_train_base, y_train_base)

y_val_pred_base = baseline_clf.predict(X_val_base)
baseline_acc = accuracy_score(y_val_base, y_val_pred_base)

print(f"Baseline VAL accuracy: {baseline_acc:.4f}")
print("\nClassification report:")
print(classification_report(y_val_base, y_val_pred_base, target_names=le.classes_))

## 5. CNN Data Preparation

For convolutional models, spectrograms are kept as 2D arrays with a channel dimension added, producing tensors of shape `(64, 130, 1)` — analogous to single-channel grayscale images.

In [ ]:
def build_xy_cnn(dataframe):
    X_list, y_list = [], []
    for _, row in dataframe.iterrows():
        spec = wav_to_logmel(row["path"])
        X_list.append(spec[..., np.newaxis])  # (64, 130, 1)
        y_list.append(row["y"])
    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.int64)

X_train_cnn, y_train_cnn = build_xy_cnn(train_df)
X_val_cnn, y_val_cnn = build_xy_cnn(val_df)

print(f"X_train: {X_train_cnn.shape}  |  X_val: {X_val_cnn.shape}")

## 6. CNN V1 — Initial Architecture

A straightforward 3-block CNN: Conv2D → ReLU → MaxPool, followed by Flatten → Dense → Dropout → Softmax. No batch normalization, no class weighting.

**Result: 0.6806** — a +40% relative improvement over baseline, but violin recall collapsed to 0.05.

In [ ]:
# V1 CNN
tf.random.set_seed(SEED)
np.random.seed(SEED)

v1_cnn = keras.Sequential([
    layers.Input(shape=(64, 130, 1)),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

v1_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

v1_cnn.summary()

In [ ]:
# Train V1
history_v1 = v1_cnn.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=20, batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True
    )],
    verbose=1
)

In [ ]:
# Evaluate V1
y_pred_v1 = np.argmax(v1_cnn.predict(X_val_cnn, verbose=0), axis=1)
v1_acc = accuracy_score(y_val_cnn, y_pred_v1)

print(f"V1 VAL accuracy: {v1_acc:.4f}")
print("\n" + classification_report(y_val_cnn, y_pred_v1, target_names=le.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v1, display_labels=le.classes_, cmap="Blues"
)
plt.title("V1 CNN — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. CNN V2 — BatchNorm + Global Average Pooling

Added **BatchNormalization** after each conv layer and replaced Flatten with **GlobalAveragePooling2D**. Also added a 4th conv block (128 filters).

**Result: 0.6825** — marginal overall improvement, but drum recall dropped to 0.00 while violin jumped to 0.96. The model learned to separate violin/guitar/piano but completely lost drum.

In [ ]:
# V2 CNN
tf.random.set_seed(SEED)
np.random.seed(SEED)

v2_cnn = keras.Sequential([
    layers.Input(shape=(64, 130, 1)),

    layers.Conv2D(16, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

v2_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Train V2
history_v2 = v2_cnn.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=25, batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True
    )],
    verbose=1
)

In [ ]:
# Evaluate V2
y_pred_v2 = np.argmax(v2_cnn.predict(X_val_cnn, verbose=0), axis=1)
v2_acc = accuracy_score(y_val_cnn, y_pred_v2)

print(f"V2 VAL accuracy: {v2_acc:.4f}")
print("\n" + classification_report(y_val_cnn, y_pred_v2, target_names=le.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v2, display_labels=le.classes_, cmap="Blues"
)
plt.title("V2 CNN — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. CNN V3 — Class Weights for Balance

Reverted to 3 conv blocks + Flatten (removing the 4th block and GAP that caused drum collapse). Added **balanced class weights** to penalize misclassification of underrepresented classes.

**Result: 0.7110** — best balance across all four classes. Violin recall recovered to 0.83. This is the `best_balanced_model`.

In [ ]:
# Compute class weights
classes = np.unique(y_train_cnn)
weights = compute_class_weight("balanced", classes=classes, y=y_train_cnn)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

for i, name in enumerate(le.classes_):
    print(f"  {name}: {class_weight_dict[i]:.4f}")

In [ ]:
# V3 CNN
tf.random.set_seed(SEED)
np.random.seed(SEED)

v3_cnn = keras.Sequential([
    layers.Input(shape=(64, 130, 1)),

    layers.Conv2D(16, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

v3_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Train V3 with class weights
history_v3 = v3_cnn.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=25, batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True
    )],
    verbose=1
)

In [ ]:
# Evaluate V3
y_pred_v3 = np.argmax(v3_cnn.predict(X_val_cnn, verbose=0), axis=1)
v3_acc = accuracy_score(y_val_cnn, y_pred_v3)

print(f"V3 VAL accuracy: {v3_acc:.4f}")
print("\n" + classification_report(y_val_cnn, y_pred_v3, target_names=le.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v3, display_labels=le.classes_, cmap="Blues"
)
plt.title("V3 CNN — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. CNN V4 — Best Validation Accuracy (0.7148)

Same architecture as V3 but with **Dropout increased to 0.5** and **ReduceLROnPlateau** added to the training callbacks. Class weights are still applied.

**Result: 0.7148** — highest overall accuracy. However, violin recall collapsed to 0.13 while drum recall rose to 0.87. This tradeoff is the key insight: a higher headline number does not always mean a better model for every class.

This is the version reported in the ACL paper as the best validation accuracy.

In [ ]:
# V4 CNN — best overall accuracy
tf.random.set_seed(SEED)
np.random.seed(SEED)

v4_cnn = keras.Sequential([
    layers.Input(shape=(64, 130, 1)),

    layers.Conv2D(16, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(), layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

v4_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

v4_cnn.summary()

In [ ]:
# Train V4 with class weights + LR scheduling
callbacks_v4 = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True
    )
]

history_v4 = v4_cnn.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=30, batch_size=32,
    callbacks=callbacks_v4,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Evaluate V4
y_pred_v4 = np.argmax(v4_cnn.predict(X_val_cnn, verbose=0), axis=1)
v4_acc = accuracy_score(y_val_cnn, y_pred_v4)

print(f"V4 VAL accuracy: {v4_acc:.4f}")
print("\n" + classification_report(y_val_cnn, y_pred_v4, target_names=le.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v4, display_labels=le.classes_, cmap="Blues"
)
plt.title("V4 CNN — Validation Confusion Matrix (Best Accuracy)")
plt.tight_layout()
plt.show()

## 10. Results Comparison

### Accuracy Progression

In [ ]:
# Final comparison
results = pd.DataFrame({
    "Version": ["Baseline", "V1", "V2", "V3", "V4"],
    "Description": [
        "LogReg on flattened spectrograms",
        "3-layer CNN, Flatten + Dense(128)",
        "+ BatchNorm, + GAP, 4 conv blocks",
        "3 conv + Flatten, + class weights",
        "+ Dropout 0.5, + ReduceLROnPlateau"
    ],
    "Val Accuracy": [baseline_acc, v1_acc, v2_acc, v3_acc, v4_acc]
})
results["Val Accuracy"] = results["Val Accuracy"].round(4)
print(results.to_string(index=False))

In [ ]:
# Accuracy bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#999999", "#5DA5DA", "#5DA5DA", "#5DA5DA", "#F15854"]
bars = ax.bar(results["Version"], results["Val Accuracy"], color=colors, edgecolor="white")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Instrument Classification — Model Progression")
ax.set_ylim(0.4, 0.78)
for bar, val in zip(bars, results["Val Accuracy"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.axhline(y=baseline_acc, color="gray", linestyle="--", alpha=0.5, label="Baseline")
ax.legend()
plt.tight_layout()
plt.show()

### V3 vs V4 — The Accuracy-Balance Tradeoff

The confusion matrices reveal that V4's higher headline accuracy comes from heavily favoring drum recall at the expense of violin. V3 is the better model if balanced per-class performance matters.

In [ ]:
# Side-by-side confusion matrices: V3 vs V4
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v3, display_labels=le.classes_,
    cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title(f"V3 — Val Acc: {v3_acc:.4f}\n(best balanced)")

ConfusionMatrixDisplay.from_predictions(
    y_val_cnn, y_pred_v4, display_labels=le.classes_,
    cmap="Reds", ax=axes[1], colorbar=False
)
axes[1].set_title(f"V4 — Val Acc: {v4_acc:.4f}\n(best overall)")

plt.suptitle("Drum ↔ Violin Tradeoff Across Model Versions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 11. Inference Demo

A simple function to classify a new audio file. In the full project, this was paired with a separate genre classifier to produce two complementary predictions (genre + instrument) for the same clip.

In [ ]:
def predict_instrument(path, model=v4_cnn, class_names=le.classes_):
    """Predict dominant instrument from an audio file."""
    spec = wav_to_logmel(path)
    x = spec[np.newaxis, ..., np.newaxis]  # (1, 64, 130, 1)

    probs = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = class_names[pred_idx]

    print(f"File: {os.path.basename(path)}")
    print(f"Predicted: {pred_label}  (confidence: {probs[pred_idx]:.4f})")
    print("\nAll probabilities:")
    for i, name in enumerate(class_names):
        bar = "█" * int(probs[i] * 30)
        print(f"  {name:8s} {probs[i]:.4f}  {bar}")

    return pred_label, probs

# Example: run on a validation sample
sample = val_df.iloc[0]
print(f"Ground truth: {sample['label']}\n")
_ = predict_instrument(sample["path"])

## 12. Save Artifacts

In [ ]:
import json

# Save model
v4_cnn.save("instrument_cnn_v4.keras")

# Save label classes
np.save("instrument_label_classes.npy", le.classes_)

# Save results summary
summary = {
    "baseline_val_acc": float(baseline_acc),
    "v1_val_acc": float(v1_acc),
    "v2_val_acc": float(v2_acc),
    "v3_val_acc": float(v3_acc),
    "v4_val_acc": float(v4_acc),
    "best_overall_model": "v4",
    "best_balanced_model": "v3"
}

with open("instrument_results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: instrument_cnn_v4.keras")
print("Saved: instrument_label_classes.npy")
print("Saved: instrument_results_summary.json")
print("\nResults:")
print(json.dumps(summary, indent=2))

---

## References

- McFee, B. et al. (2015). *librosa: Audio and music signal analysis in Python.* Proc. 14th Python in Science Conf.
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine learning in Python.* JMLR 12:2825–2830.
- Prasad, S. (2021). *Musical Instrument's Sound Dataset.* Kaggle.
- TensorFlow Developers (2023). *TensorFlow.* https://www.tensorflow.org/